# Create Letten Prize Awards

Creates OpenAlex award rows from the official Letten Prize WordPress REST records.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/letten_prize_to_s3.py` to download the official WordPress REST records, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:** `https://lettenprize.com/wp-json/wp/v2/posts` and `https://lettenprize.com/wp-json/wp/v2/pages`  
**S3 location:** `s3a://openalex-ingest/awards/letten_prize/letten_prize_laureates.parquet`  
**Local full-source validation on 2026-05-28:** 4 Letten Prize laureate rows for 2018, 2021, 2023, and 2025; 100% native ID/laureate/title/year/date/amount/currency/affiliation/description/landing-page coverage; NOK 9,000,000 total.

**Letten Foundation funder:**
- funder_id: 4320328141
- display_name: "Letten Foundation"
- country: NO

**Source-shape note:** Letten Prize is a partnership between the Letten Foundation and the Young Academy of Norway. OpenAlex has a Letten Foundation funder row, so these prize rows map to Letten Foundation. Runners-up/finalists are excluded because this prize-pattern ingest is one row per laureate.

**Amount/currency decision:** The source publishes explicit NOK prize amounts. 2018 and 2021 use NOK 2,000,000 from official winner/announcement pages. 2023 and 2025 use NOK 2,500,000 from official call/current criteria pages. `amount` is total prize value; `amount_note` preserves the source rule.

**Mapping summary:** One OpenAlex award row per Letten Prize laureate. Native key is `letten-prize-{year}-{laureate_slug}`. The laureate is mapped to `lead_investigator`; source affiliation is mapped to `lead_investigator.affiliation.name`.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.letten_prize_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/letten_prize/letten_prize_laureates.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 4 rows.
SELECT COUNT(*) as total_letten_prize_laureates
FROM openalex.awards.letten_prize_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.letten_prize_raw;

In [ ]:
%sql
-- Sample raw Letten Prize data.
SELECT
    funder_award_id,
    laureate_name,
    affiliation,
    award_year,
    award_date,
    amount,
    currency,
    amount_note,
    landing_page_url
FROM openalex.awards.letten_prize_raw
ORDER BY award_year;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'letten_prize_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|obligated|outlay|grant|award|donation|support|stipend|prize';

In [ ]:
%sql
-- Confirm amount/date/name coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(funder_award_id) AS rows_with_native_id,
    COUNT(laureate_name) AS rows_with_laureate_name,
    COUNT(affiliation) AS rows_with_affiliation,
    ROUND(try_divide(COUNT(affiliation), COUNT(*)) * 100.0, 1) AS pct_affiliation,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_amount_nok,
    COUNT(award_date) AS rows_with_award_date,
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year,
    collect_set(currency) AS currencies
FROM openalex.awards.letten_prize_raw;

In [ ]:
%sql
-- Native-key inspection. funder_award_id must be unique.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_record_id) AS distinct_source_record_ids,
    COUNT(DISTINCT laureate_name) AS distinct_laureates
FROM openalex.awards.letten_prize_raw;

In [ ]:
%sql
-- Award-year and amount distribution.
SELECT award_year, amount, currency, laureate_name, affiliation, landing_page_url
FROM openalex.awards.letten_prize_raw
ORDER BY TRY_CAST(award_year AS INT);

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return TRUE. If this fails, STOP: Letten Foundation is missing from openalex.common.funder.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Letten Foundation F4320328141'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320328141;

In [ ]:
%sql
-- Inspect the funder row used for the award struct.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320328141;

## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.letten_prize_awards
USING delta
AS
WITH
letten_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320328141
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'NOK' ELSE NULL END AS parsed_currency,
        TRY_TO_DATE(award_date, 'yyyy-MM-dd') AS parsed_award_date,
        TRY_CAST(award_year AS INT) AS parsed_award_year
    FROM openalex.awards.letten_prize_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
      AND laureate_name IS NOT NULL
      AND TRIM(laureate_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'prize' as funding_type,
        'Letten Prize' as funder_scheme,

        'letten_prize' as provenance,

        g.parsed_award_date as start_date,
        g.parsed_award_date as end_date,
        g.parsed_award_year as start_year,
        g.parsed_award_year as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_award_date as role_start,
            struct(
                NULLIF(TRIM(g.affiliation), '') as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN letten_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 176

In [ ]:
%sql
-- Remove previous Letten Prize data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'letten_prize' AND priority = 176;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    176 as priority
FROM openalex.awards.letten_prize_awards;

## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_letten_prize_awards
FROM openalex.awards.letten_prize_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.letten_prize_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    lead_investigator.affiliation.name AS source_affiliation,
    landing_page_url
FROM openalex.awards.letten_prize_awards
ORDER BY start_year;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.letten_prize_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only Letten Foundation.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.letten_prize_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness and amount coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(currency) as has_currency,
    COUNT(start_date) as has_start_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.given_name) as has_given_name,
    COUNT(lead_investigator.family_name) as has_family_name,
    COUNT(lead_investigator.affiliation.name) as has_source_affiliation,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(SUM(amount), 0) as total_nok
FROM openalex.awards.letten_prize_awards;

In [ ]:
%sql
-- Amount/currency fail-fast check for this prize pattern.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.letten_prize_awards;

In [ ]:
%sql
-- Year and laureate distribution.
SELECT start_year, display_name, amount, currency, lead_investigator.affiliation.name AS source_affiliation
FROM openalex.awards.letten_prize_awards
ORDER BY start_year;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 176.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    ROUND(SUM(amount), 0) as total_nok
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'letten_prize' AND priority = 176
GROUP BY priority, provenance;

## Handoff / Admin Notes

Contractor-complete handoff only. The contractor has no S3 or Databricks access, so an admin must:

1. Run `scripts/local/letten_prize_to_s3.py` to upload `s3://openalex-ingest/awards/letten_prize/letten_prize_laureates.parquet`.
2. Run this notebook on Databricks.
3. Run the Step 6 verification cells and QA the inserted `provenance='letten_prize'`, `priority=176` rows.
4. Mark the tracker row Complete after production API verification.

Do not add job YAML until an admin has run and verified the ingest.